# RAFT Fine-Tuning v5 — Explicit Oracle Labels

Notebook ini memperbaiki pipeline sebelumnya dengan prinsip berikut:

1. `selected_doc_ids` dan `rejected_doc_ids` dibaca langsung dari dataset—tidak dihitung melalui lexical overlap.
2. Nama atau prefix `document_id` tidak pernah digunakan sebagai sinyal pemilihan konteks.
3. Prompt training sama dengan prompt pada `raft_service`.
4. Loss hanya dihitung pada respons assistant (*assistant-only loss*).
5. Split train/eval dilakukan berdasarkan `source_group` untuk mengurangi kebocoran sumber yang sama.
6. `ground_truth` tidak digunakan dalam prompt atau generation; ia tetap berada di sisi evaluasi aplikasi lokal.

Format dataset yang diharapkan adalah RAFT v5 dengan field minimal:
`instruction`, `documents`, `thought_process`, `completion`, `selected_doc_ids`, `rejected_doc_ids`, `answerability`, dan `source_group`.

## 1. Konfigurasi dan pengecekan lingkungan

In [1]:
import os
import gc
import json
import random
import re
from pathlib import Path

import torch
import transformers
import trl
from datasets import load_dataset, DatasetDict

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = "/workspace/model/Meta-Llama-3.1-8B-Instruct"
DATASET_PATH = "../data/dataset/raft_dataset_finalv5_corrected.jsonl"
OUTPUT_DIR = "/workspace/model/outputs_lora_raft_v5"
ADAPTER_DIR = "/workspace/model/lora_adapter_raft_v5"
MERGED_DIR = "/workspace/model/model_merged_raft_v5"

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = False
DTYPE = None

print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL         :", trl.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GiB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

assert torch.cuda.is_available(), "CUDA tidak tersedia."
assert Path(BASE_MODEL).exists(), f"Base model tidak ditemukan: {BASE_MODEL}"
assert Path(DATASET_PATH).exists(), f"Dataset tidak ditemukan: {DATASET_PATH}"

/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch     : 2.11.0+cu128
Transformers: 5.5.0
TRL         : 0.24.0
CUDA        : True
GPU         : NVIDIA B200
VRAM (GiB)  : 178.36


## 2. Load base model dan pasang LoRA

In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

print("Base model dan LoRA berhasil dimuat.")

/workspace/.venv/lib/python3.10/site-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.361 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 10.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 108.17it/s]
Unsloth: Will load /workspace/model/Meta-Llama-3.1-8B-Instruct as a legacy tokenizer.


/workspace/model/Meta-Llama-3.1-8B-Instruct does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Base model dan LoRA berhasil dimuat.


## 3. Satu system prompt untuk training dan inference

In [3]:
RAFT_SYSTEM_PROMPT = """Anda adalah asisten AI ahli dalam menjawab pertanyaan berdasarkan dokumen hukum dan peraturan desa.

Tugas Anda:
1. Periksa semua dokumen referensi yang diberikan.
2. Pilih hanya dokumen yang benar-benar menjawab pertanyaan.
3. Cocokkan secara teliti nama desa, nomor dan tahun peraturan, pasal, ayat, serta isi ketentuan yang ditanyakan.
4. Abaikan dokumen yang tidak relevan, salah desa, salah peraturan, salah pasal/ayat, atau hanya mirip topiknya.
5. Jika beberapa dokumen tampak serupa tetapi rincian isinya berbeda, jangan menggabungkan isinya. Pilih hanya dokumen yang benar-benar sesuai dengan pertanyaan.
6. Jangan menambah, mengganti, atau menyimpulkan rincian yang tidak tertulis dalam dokumen yang dipilih.
7. Jika tidak ada dokumen valid, katakan bahwa informasi tidak ditemukan pada dokumen yang diberikan.
8. Jawaban akhir hanya boleh berdasarkan dokumen yang dipilih.

Format jawaban HARUS seperti ini:

KONTEKS_DIPILIH: [id dokumen]
KONTEKS_DITOLAK: [id dokumen]

<thought>
alasan singkat memilih dan menolak dokumen
</thought>

JAWABAN:
<jawaban akhir>"""

BEGIN_TEXT = "<|begin_of_text|>"
SYSTEM_HEADER = "<|start_header_id|>system<|end_header_id|>"
USER_HEADER = "<|start_header_id|>user<|end_header_id|>"
ASSISTANT_HEADER = "<|start_header_id|>assistant<|end_header_id|>"
EOT = "<|eot_id|>"


def format_documents(documents):
    return "\n\n".join(f"[{i}] {doc}" for i, doc in enumerate(documents, start=1))


def build_user_prompt(instruction, documents):
    return (
        f"{BEGIN_TEXT}{SYSTEM_HEADER}\n\n"
        f"{RAFT_SYSTEM_PROMPT}{EOT}{USER_HEADER}\n\n"
        f"Pertanyaan: {instruction.strip()}\n\n"
        f"Dokumen Referensi:\n{format_documents(documents)}\n\n"
        f"{EOT}{ASSISTANT_HEADER}\n\n"
    )

print(RAFT_SYSTEM_PROMPT)

Anda adalah asisten AI ahli dalam menjawab pertanyaan berdasarkan dokumen hukum dan peraturan desa.

Tugas Anda:
1. Periksa semua dokumen referensi yang diberikan.
2. Pilih hanya dokumen yang benar-benar menjawab pertanyaan.
3. Cocokkan secara teliti nama desa, nomor dan tahun peraturan, pasal, ayat, serta isi ketentuan yang ditanyakan.
4. Abaikan dokumen yang tidak relevan, salah desa, salah peraturan, salah pasal/ayat, atau hanya mirip topiknya.
5. Jika beberapa dokumen tampak serupa tetapi rincian isinya berbeda, jangan menggabungkan isinya. Pilih hanya dokumen yang benar-benar sesuai dengan pertanyaan.
6. Jangan menambah, mengganti, atau menyimpulkan rincian yang tidak tertulis dalam dokumen yang dipilih.
7. Jika tidak ada dokumen valid, katakan bahwa informasi tidak ditemukan pada dokumen yang diberikan.
8. Jawaban akhir hanya boleh berdasarkan dokumen yang dipilih.

Format jawaban HARUS seperti ini:

KONTEKS_DIPILIH: [id dokumen]
KONTEKS_DITOLAK: [id dokumen]

<thought>
alasan si

## 4. Load dan validasi dataset RAFT v5

In [4]:
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print("Total data:", len(raw_dataset))
print("Kolom     :", raw_dataset.column_names)

REQUIRED_COLUMNS = {
    "instruction",
    "documents",
    "thought_process",
    "completion",
    "selected_doc_ids",
    "rejected_doc_ids",
    "answerability",
    "source_group",
}
missing = REQUIRED_COLUMNS - set(raw_dataset.column_names)
assert not missing, f"Kolom wajib belum ada: {sorted(missing)}"

Generating train split: 425 examples [00:00, 51108.99 examples/s]

Total data: 425
Kolom     : ['instruction', 'documents', 'thought_process', 'completion', 'record_id', 'selected_doc_ids', 'rejected_doc_ids', 'answerability', 'document_judgments', 'source_group']


In [5]:
def normalize_ids(value):
    if value is None:
        return []
    if isinstance(value, list):
        return sorted({int(x) for x in value})
    if isinstance(value, (int, float)):
        return [int(value)]
    if isinstance(value, str):
        return sorted({int(x) for x in re.findall(r"\d+", value)})
    raise TypeError(f"Tipe ID tidak didukung: {type(value)}")


def validate_record(row, index):
    errors = []
    docs = row.get("documents")
    selected = normalize_ids(row.get("selected_doc_ids"))
    rejected = normalize_ids(row.get("rejected_doc_ids"))

    if not isinstance(row.get("instruction"), str) or not row["instruction"].strip():
        errors.append("instruction kosong")
    if not isinstance(docs, list) or not docs:
        errors.append("documents harus list dan tidak kosong")
        docs = []
    if not isinstance(row.get("completion"), str) or not row["completion"].strip():
        errors.append("completion kosong")

    valid_ids = set(range(1, len(docs) + 1))
    selected_set, rejected_set = set(selected), set(rejected)

    if not selected_set.issubset(valid_ids):
        errors.append(f"selected_doc_ids di luar rentang: {selected}")
    if not rejected_set.issubset(valid_ids):
        errors.append(f"rejected_doc_ids di luar rentang: {rejected}")
    if selected_set & rejected_set:
        errors.append("dokumen yang sama masuk selected dan rejected")
    if selected_set | rejected_set != valid_ids:
        errors.append("selected + rejected harus mencakup seluruh dokumen")

    answerability = str(row.get("answerability", "")).lower()
    if answerability in {"full", "partial"} and not selected:
        errors.append(f"answerability={answerability} tetapi selected kosong")
    if answerability in {"none", "unanswerable", "absent"} and selected:
        errors.append(f"answerability={answerability} tetapi selected tidak kosong")

    judgments = row.get("document_judgments")
    if isinstance(judgments, list) and judgments:
        labels = {}
        for item in judgments:
            try:
                labels[int(item["document"])] = str(item["label"]).lower()
            except Exception:
                errors.append("format document_judgments tidak valid")
                break
        for doc_id in selected:
            if labels.get(doc_id) not in {"selected", "oracle"}:
                errors.append(f"judgment dokumen {doc_id} tidak konsisten dengan selected")
        for doc_id in rejected:
            if labels.get(doc_id) not in {"rejected", "distractor"}:
                errors.append(f"judgment dokumen {doc_id} tidak konsisten dengan rejected")

    return errors

all_errors = []
for idx, row in enumerate(raw_dataset):
    errors = validate_record(row, idx)
    if errors:
        all_errors.append({"row": idx, "record_id": row.get("record_id"), "errors": errors})

print("Jumlah record invalid:", len(all_errors))
if all_errors:
    print(json.dumps(all_errors[:20], ensure_ascii=False, indent=2))
    raise ValueError("Dataset belum valid. Perbaiki record yang ditampilkan sebelum training.")

print("Semua record lolos validasi struktur dan label.")

Jumlah record invalid: 0
Semua record lolos validasi struktur dan label.


> Validasi di atas tidak membaca atau menafsirkan nama `document_id`. Seluruh target konteks berasal hanya dari `selected_doc_ids` dan `rejected_doc_ids` yang telah ditentukan saat pembuatan dataset.

## 5. Bentuk target assistant dari label eksplisit

In [6]:
def format_thought_process(thought_process):
    if thought_process is None:
        return "Tidak ada dokumen yang dapat dijadikan dasar jawaban."

    if isinstance(thought_process, str):
        return thought_process.strip()

    if isinstance(thought_process, dict):
        lines = []
        for item in thought_process.get("document_analysis", []) or []:
            doc_id = item.get("document", "?")
            analysis = str(item.get("analysis", "")).strip()
            if analysis:
                lines.append(f"Dokumen {doc_id}: {analysis}")
        summary = str(thought_process.get("summary", "")).strip()
        if summary:
            lines.append(f"Ringkasan: {summary}")
        return "\n".join(lines).strip()

    return str(thought_process).strip()


def ids_to_text(ids):
    ids = normalize_ids(ids)
    return ", ".join(map(str, ids)) if ids else "-"


def build_assistant_response(row):
    return (
        f"KONTEKS_DIPILIH: {ids_to_text(row['selected_doc_ids'])}\n"
        f"KONTEKS_DITOLAK: {ids_to_text(row['rejected_doc_ids'])}\n\n"
        f"<thought>\n{format_thought_process(row['thought_process'])}\n</thought>\n\n"
        f"JAWABAN:\n{row['completion'].strip()}"
    )


def formatting_prompts_func(examples):
    texts = []
    batch_size = len(examples["instruction"])

    for i in range(batch_size):
        row = {column: examples[column][i] for column in examples.keys()}
        prompt = build_user_prompt(row["instruction"], row["documents"])
        response = build_assistant_response(row)
        texts.append(prompt + response + tokenizer.eos_token)

    return {"text": texts}

formatted_dataset = raw_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=raw_dataset.column_names,
    desc="Formatting RAFT v5",
)

print(formatted_dataset[0]["text"][:8000])

Formatting RAFT v5: 100%|██████████| 425/425 [00:00<00:00, 15417.97 examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI ahli dalam menjawab pertanyaan berdasarkan dokumen hukum dan peraturan desa.

Tugas Anda:
1. Periksa semua dokumen referensi yang diberikan.
2. Pilih hanya dokumen yang benar-benar menjawab pertanyaan.
3. Cocokkan secara teliti nama desa, nomor dan tahun peraturan, pasal, ayat, serta isi ketentuan yang ditanyakan.
4. Abaikan dokumen yang tidak relevan, salah desa, salah peraturan, salah pasal/ayat, atau hanya mirip topiknya.
5. Jika beberapa dokumen tampak serupa tetapi rincian isinya berbeda, jangan menggabungkan isinya. Pilih hanya dokumen yang benar-benar sesuai dengan pertanyaan.
6. Jangan menambah, mengganti, atau menyimpulkan rincian yang tidak tertulis dalam dokumen yang dipilih.
7. Jika tidak ada dokumen valid, katakan bahwa informasi tidak ditemukan pada dokumen yang diberikan.
8. Jawaban akhir hanya boleh berdasarkan dokumen yang dipilih.

Format jawaban HARUS seperti ini:

KONTEKS_DIPILIH: [i

## 6. Audit target pada contoh Cipedes

In [7]:
def find_record_by_id(dataset, record_id):
    for row in dataset:
        if row.get("record_id") == record_id:
            return row
    return None

example = find_record_by_id(raw_dataset, "raft-v5-0246")
if example is not None:
    print(build_user_prompt(example["instruction"], example["documents"]))
    print("----- TARGET ASSISTANT -----")
    print(build_assistant_response(example))
    assert ids_to_text(example["selected_doc_ids"]) == "1"
    assert ids_to_text(example["rejected_doc_ids"]) == "2, 3, 4"
else:
    print("Record raft-v5-0246 tidak ditemukan; audit menggunakan record pertama.")
    example = raw_dataset[0]
    print(build_assistant_response(example))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI ahli dalam menjawab pertanyaan berdasarkan dokumen hukum dan peraturan desa.

Tugas Anda:
1. Periksa semua dokumen referensi yang diberikan.
2. Pilih hanya dokumen yang benar-benar menjawab pertanyaan.
3. Cocokkan secara teliti nama desa, nomor dan tahun peraturan, pasal, ayat, serta isi ketentuan yang ditanyakan.
4. Abaikan dokumen yang tidak relevan, salah desa, salah peraturan, salah pasal/ayat, atau hanya mirip topiknya.
5. Jika beberapa dokumen tampak serupa tetapi rincian isinya berbeda, jangan menggabungkan isinya. Pilih hanya dokumen yang benar-benar sesuai dengan pertanyaan.
6. Jangan menambah, mengganti, atau menyimpulkan rincian yang tidak tertulis dalam dokumen yang dipilih.
7. Jika tidak ada dokumen valid, katakan bahwa informasi tidak ditemukan pada dokumen yang diberikan.
8. Jawaban akhir hanya boleh berdasarkan dokumen yang dipilih.

Format jawaban HARUS seperti ini:

KONTEKS_DIPILIH: [i

## 7. Split berdasarkan `source_group`

In [8]:
def group_train_eval_split(dataset, eval_ratio=0.20, seed=SEED):
    groups = sorted({str(x) for x in dataset["source_group"]})
    rng = random.Random(seed)
    rng.shuffle(groups)

    eval_group_count = max(1, round(len(groups) * eval_ratio))
    eval_groups = set(groups[:eval_group_count])

    train_indices = [i for i, g in enumerate(dataset["source_group"]) if str(g) not in eval_groups]
    eval_indices = [i for i, g in enumerate(dataset["source_group"]) if str(g) in eval_groups]

    assert train_indices, "Train split kosong."
    assert eval_indices, "Eval split kosong."
    assert not (
        set(dataset.select(train_indices)["source_group"])
        & set(dataset.select(eval_indices)["source_group"])
    ), "Terjadi kebocoran source_group antara train dan eval."

    return train_indices, eval_indices, eval_groups

train_indices, eval_indices, eval_groups = group_train_eval_split(raw_dataset)
train_dataset = formatted_dataset.select(train_indices)
eval_dataset = formatted_dataset.select(eval_indices)

print("Jumlah source_group :", len(set(raw_dataset["source_group"])))
print("Train rows          :", len(train_dataset))
print("Eval rows           :", len(eval_dataset))
print("Eval groups         :", len(eval_groups))

Jumlah source_group : 281
Train rows          : 334
Eval rows           : 91
Eval groups         : 56


## 8. Konfigurasi SFT dan assistant-only loss

In [9]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    dataset_text_field="text",
    dataset_num_proc=1,
    packing=False,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,

    learning_rate=5e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    logging_steps=10,
    optim="adamw_8bit",
    lr_scheduler_type="linear",
    seed=SEED,

    output_dir=OUTPUT_DIR,
    logging_dir=f"{OUTPUT_DIR}/tensorboard_logs",
    report_to="tensorboard",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part=f"{USER_HEADER}\n\n",
    response_part=f"{ASSISTANT_HEADER}\n\n",
)

print("SFTTrainer siap dengan assistant-only loss.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Unsloth: Tokenizing ["text"] (num_proc=1): 100%|██████████| 91/91 [00:00<00:00, 108.68 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Filter (num_proc=64): 100%|██████████| 91/91 [00:01<00:00, 64.45 examples/s]

SFTTrainer siap dengan assistant-only loss.


## 9. Verifikasi token yang benar-benar dilatih

In [10]:
def decode_trainable_labels(sample):
    labels = sample["labels"]
    token_ids = [
        token_id for token_id, label in zip(sample["input_ids"], labels)
        if label != -100
    ]
    return tokenizer.decode(token_ids, skip_special_tokens=False)

trainable_text = decode_trainable_labels(trainer.train_dataset[0])
print(trainable_text[:6000])

assert "KONTEKS_DIPILIH:" in trainable_text
assert "JAWABAN:" in trainable_text
assert "Pertanyaan:" not in trainable_text, (
    "Masking gagal: bagian user masih ikut dihitung loss."
)
print("Masking assistant-only terverifikasi.")

KONTEKS_DIPILIH: 1, 2, 3
KONTEKS_DITOLAK: -

<thought>
Dokumen 1: Dokumen dipilih karena memuat ketentuan yang digunakan secara langsung dalam jawaban.
Dokumen 2: Dokumen dipilih karena memuat ketentuan yang digunakan secara langsung dalam jawaban.
Dokumen 3: Dokumen dipilih karena memuat ketentuan yang digunakan secara langsung dalam jawaban.
Ringkasan: Dokumen yang dipilih: 1, 2, 3. Dokumen yang ditolak: tidak ada.
</thought>

JAWABAN:
Pemerintahan desa adalah penyelenggaraan urusan pemerintahan oleh pemerintah desa dan badan permusyawaratan desa dalam mengatur dan mengurus kepentingan masyarakat setempat berdasarkan asal-usul dan adat istiadat setempat yang diakui dan dihormati dalam sistem pemerintahan negara kesatuan republik indonesia.<|eot_id|>
Masking assistant-only terverifikasi.


## 10. Training

In [11]:
print("Memulai training...")
trainer_stats = trainer.train()
trainer.save_state()

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/train_metrics.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)

print("Training selesai.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval loss :", trainer.state.best_metric)
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Memulai training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 334 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.375732,0.173828
2,0.142444,0.091774
3,0.083473,0.084369


/workspace/.venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/workspace/.venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/workspace/.venv/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Pleas

Training selesai.
Best checkpoint: /workspace/model/outputs_lora_raft_v5/checkpoint-33
Best eval loss : 0.08436927944421768
TrainOutput(global_step=33, training_loss=0.19020399270635663, metrics={'train_runtime': 545.2828, 'train_samples_per_second': 1.838, 'train_steps_per_second': 0.061, 'total_flos': 7.800828866750054e+16, 'train_loss': 0.19020399270635663, 'epoch': 3.0})


## 11. Simpan adapter dan manifest

In [12]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

manifest = {
    "training_version": "raft_v5_explicit_labels",
    "base_model": BASE_MODEL,
    "adapter_dir": ADAPTER_DIR,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "system_prompt": RAFT_SYSTEM_PROMPT,
    "label_source": "selected_doc_ids_and_rejected_doc_ids",
    "document_id_used_as_training_signal": False,
    "ground_truth_used_for_generation": False,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
}
with open(f"{ADAPTER_DIR}/raft_training_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Adapter tersimpan:", ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in /workspace/model/lora_adapter_raft_v5/tokenizer_config.json.


Adapter tersimpan: /workspace/model/lora_adapter_raft_v5


### Opsional: merge 16-bit

`raft_service` Anda sudah dapat memuat `ADAPTER_DIR`, sehingga merge tidak wajib. Jalankan cell berikut hanya jika aplikasi akan memuat model merged.

In [ ]:
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model tersimpan:", MERGED_DIR)

## 12. Inference smoke test dengan kontrak yang sama

In [ ]:
FastLanguageModel.for_inference(model)


def generate_answer(question, documents, max_new_tokens=1024):
    prompt = build_user_prompt(question, documents)
    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0, input_length:]
    result = tokenizer.decode(generated_ids, skip_special_tokens=False)
    return (
        result.replace("<|eot_id|>", "")
        .replace("<|end_of_text|>", "")
        .strip()
    )

if example is not None:
    result = generate_answer(example["instruction"], example["documents"])
    print(result)

## 13. Uji invariansi urutan dokumen

In [ ]:
def remap_expected_ids(original_docs, reordered_docs, original_selected):
    selected_texts = {original_docs[i - 1] for i in normalize_ids(original_selected)}
    return [i for i, doc in enumerate(reordered_docs, start=1) if doc in selected_texts]


def run_order_tests(row):
    original_docs = list(row["documents"])
    permutations = {
        "original": original_docs,
        "reversed": list(reversed(original_docs)),
    }
    if len(original_docs) > 2:
        shifted = original_docs[1:] + original_docs[:1]
        permutations["shifted"] = shifted

    for name, docs in permutations.items():
        expected = remap_expected_ids(original_docs, docs, row["selected_doc_ids"])
        print("=" * 80)
        print("Order   :", name)
        print("Expected:", expected)
        print(generate_answer(row["instruction"], docs))

if example is not None:
    run_order_tests(example)

## 14. Bersihkan VRAM setelah selesai

In [ ]:
# Jalankan hanya setelah seluruh proses save dan pengujian selesai.
# del trainer, model
# gc.collect()
# torch.cuda.empty_cache()
# print("VRAM dibersihkan.")